# National LMIS — 04: Feature Engineering for Time-Series Forecasting
**Ministry of Skill Development & Entrepreneurship (MSDE) | SIH_26246**

This notebook engineers historical lag, rolling average, and calendar features for the supervised demand forecaster:
- **Lags**: `demand_lag_1`, `demand_lag_2`, `demand_lag_3`
- **Rolling Window**: `rolling_mean_3` (closed left to strictly prevent look-ahead leakage)
- **Temporal**: `month_num`, `quarter`, `year`, `hiring_growth`


In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
df_jobs = pd.read_csv(PROCESSED_DIR / 'cleaned_job_postings.csv')


### 1. Construct Chronological Features


In [ ]:
data = df_jobs.copy()
data['month_dt'] = pd.to_datetime(data['month'], format='%Y-%m')
data = data.sort_values(by=['state', 'district', 'trade', 'month_dt']).reset_index(drop=True)
group_keys = ['state', 'district', 'sector', 'trade']

# Lags
data['demand_lag_1'] = data.groupby(group_keys)['job_postings'].shift(1)
data['demand_lag_2'] = data.groupby(group_keys)['job_postings'].shift(2)
data['demand_lag_3'] = data.groupby(group_keys)['job_postings'].shift(3)

# Rolling mean
data['rolling_mean_3'] = data.groupby(group_keys)['job_postings'].apply(lambda s: s.shift(1).rolling(3, min_periods=1).mean()).reset_index(drop=True)

# Calendar features
data['month_num'] = data['month_dt'].dt.month
data['quarter'] = data['month_dt'].dt.quarter
data['year'] = data['month_dt'].dt.year
data['hiring_growth'] = pd.to_numeric(data['hiring_growth'], errors='coerce').fillna(0.0)

featured = data.dropna(subset=['demand_lag_1', 'demand_lag_2', 'demand_lag_3']).copy()
print(f'Total usable featured records: {len(featured)}')
display(featured[['trade', 'month', 'job_postings', 'demand_lag_1', 'rolling_mean_3', 'hiring_growth']].head(8))


### 2. Feature Correlation Matrix


In [ ]:
corr = featured[['job_postings', 'demand_lag_1', 'demand_lag_2', 'demand_lag_3', 'rolling_mean_3', 'hiring_growth']].corr()
print('Correlation with target variable (job_postings):')
print(corr['job_postings'].sort_values(ascending=False))


### 3. Save Engineered Dataset


In [ ]:
featured.drop(columns=['month_dt'], inplace=True, errors='ignore')
featured.to_csv(PROCESSED_DIR / 'forecast_dataset.csv', index=False)
print(f'Saved forecast_dataset.csv with {len(featured)} rows.')
